# Construction Project Risk Pipeline

**Author:** Kenesuo Wolowolo  
**Dataset:** 1,300-row construction project task dataset  
**Tools:** Python, pandas  
**GitHub:** [github.com/Kennyg-w](https://github.com/Kennyg-w)

---

## Project Overview

This notebook builds a 5-stage data pipeline to analyse construction project tasks, 
engineer risk features, flag critical tasks, and produce a project manager-ready summary report.

### Pipeline Stages
1. **Load & Validate** — schema checks, missing values, range assertions
2. **Feature Engineering** — cost per day, complexity score, constraint scores
3. **Risk Analysis** — composite risk index, underestimated risk detection
4. **Critical Task Flagging** — priority scoring, tier assignment, triple threat detection
5. **Summary Report & Export** — PM-ready report, enriched CSV export

### Key Findings
- **$64.4M** total material cost across 1,300 tasks
- **8 P1 tasks** requiring immediate attention
- **87 tasks** flagged as underestimated risk (labelled Low, high complexity)
- **5 triple threat tasks** — high cost, high risk, schedule critical simultaneously


## Setup & Imports

In [30]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


OUTPUT_CSV = 'construction_pipeline_output.csv'
OUTPUT_REPORT = 'construction_risk_report.txt'

---
## Stage 1 — Load & Validate

Before any analysis, confirm the dataset matches the expected schema. 
Assertions raise errors immediately if something is wrong — 
catching issues at ingestion rather than downstream.


In [31]:
# Load dataset
df = pd.read_csv('../data/raw/construction_dataset.csv')

# 1. Check all expected columns are present
expected_columns = [
    'Task_ID', 'Task_Duration_Days', 'Labor_Required',
    'Equipment_Units', 'Material_Cost_USD', 'Start_Constraint',
    'Risk_Level', 'Resource_Constraint_Score',
    'Site_Constraint_Score', 'Dependency_Count'
]
missing_cols = [c for c in expected_columns if c not in df.columns]
assert len(missing_cols) == 0, f'Missing columns: {missing_cols}'
print('✅ All expected columns present')

# 2. Check dataset is not empty
assert len(df) > 0, 'Dataset is empty'
print(f'✅ Row count: {len(df):,} rows')

# 3. Check for missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print('✅ No missing values found')
else:
    print('⚠️  Missing values detected:')
    print(missing[missing > 0])

# 4. Check Risk_Level contains only valid values
valid_risk = {'Low', 'Medium', 'High'}
actual_risk = set(df['Risk_Level'].unique())
assert actual_risk <= valid_risk, f'Unexpected Risk_Level values: {actual_risk - valid_risk}'
print(f'✅ Risk_Level values valid: {actual_risk}')

# 5. Check numeric columns are within sensible ranges
assert df['Task_Duration_Days'].min() > 0, 'Task duration must be positive'
assert df['Material_Cost_USD'].min() >= 0, 'Cost cannot be negative'
assert df['Resource_Constraint_Score'].between(0, 1).all(), 'Resource score must be 0-1'
assert df['Site_Constraint_Score'].between(0, 1).all(), 'Site score must be 0-1'
print('✅ Numeric ranges all valid')

# 6. Check Task_ID is unique
assert df['Task_ID'].nunique() == len(df), 'Task_IDs are not unique'
print('✅ Task_IDs are all unique')

print()
print('--- DATASET SUMMARY ---')
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')
print()
print('Risk breakdown:')
print(df['Risk_Level'].value_counts().to_string())

✅ All expected columns present
✅ Row count: 1,300 rows
✅ No missing values found
✅ Risk_Level values valid: {'Low', 'High', 'Medium'}
✅ Numeric ranges all valid
✅ Task_IDs are all unique

--- DATASET SUMMARY ---
Rows: 1,300   Columns: 10

Risk breakdown:
Risk_Level
Low       660
Medium    377
High      263


---
## Stage 2 — Feature Engineering

Derive new columns that add analytical value beyond the raw data. 
Each feature is designed to capture a different dimension of task complexity or cost.

| Feature | Formula | Purpose |
|---|---|---|
| `Cost_Per_Day` | Cost ÷ Duration | Reveals expensive short tasks |
| `Labour_Intensity` | Workers ÷ Equipment | Labour vs equipment dependency |
| `Combined_Constraint_Score` | (Resource + Site) ÷ 2 | Single constraint metric |
| `Complexity_Score` | Weighted: duration + dependencies + constraints | Overall task difficulty |
| `Risk_Score` | Low=1, Medium=2, High=3 | Numeric risk for calculations |


In [32]:
# Cost per day — how expensive relative to duration
df['Cost_Per_Day'] = (df['Material_Cost_USD'] / df['Task_Duration_Days']).round(2)

# Labour intensity — workers per equipment unit
df['Labour_Intensity'] = (df['Labor_Required'] / df['Equipment_Units'].replace(0, 1)).round(2)

# Combined constraint score — average of resource and site scores
df['Combined_Constraint_Score'] = (
    (df['Resource_Constraint_Score'] + df['Site_Constraint_Score']) / 2
).round(3)

# Complexity score — weighted combination of duration, dependencies, constraints
df['Complexity_Score'] = (
    (df['Task_Duration_Days'] / df['Task_Duration_Days'].max() * 0.4) +
    (df['Dependency_Count'] / df['Dependency_Count'].max() * 0.3) +
    (df['Combined_Constraint_Score'] * 0.3)
).round(3)

# Risk score — convert text label to numeric
risk_map = {'Low': 1, 'Medium': 2, 'High': 3}
df['Risk_Score'] = df['Risk_Level'].map(risk_map)

# Preview new features
new_cols = ['Task_ID', 'Cost_Per_Day', 'Labour_Intensity',
            'Combined_Constraint_Score', 'Complexity_Score', 'Risk_Score']
print('New features (first 10 rows):')
print(df[new_cols].head(10).to_string(index=False))
print()
print('Feature ranges:')
print(df[['Cost_Per_Day','Labour_Intensity','Complexity_Score']].describe().round(2).to_string())

New features (first 10 rows):
Task_ID  Cost_Per_Day  Labour_Intensity  Combined_Constraint_Score  Complexity_Score  Risk_Score
     T1        322.88              2.33                       0.50              0.68           2
     T2       1125.72              1.00                       0.46              0.43           1
     T3        110.82             11.00                       0.69              0.60           1
     T4        317.69              0.20                       0.54              0.74           1
     T5       3178.94              3.80                       0.74              0.54           1
     T6       1063.70              2.00                       0.42              0.65           3
     T7        937.91              3.17                       0.48              0.54           1
     T8        917.71              1.22                       0.33              0.51           1
     T9        159.10              4.00                       0.56              0.51           1


---
## Stage 3 — Risk Analysis

Combine the engineered features to produce a **Composite Risk Index** 
and flag tasks that warrant further investigation.

Three flags are raised:
- **Cost_Risk_Flag** — top quartile cost AND labelled High risk
- **Underestimated_Risk_Flag** — high complexity but labelled Low risk
- **Schedule_Critical_Flag** — 3+ dependencies AND long duration


In [33]:
# Composite risk index — weighted combination of risk, complexity, constraints
df['Composite_Risk_Index'] = (
    (df['Risk_Score'] / 3 * 0.5) +
    (df['Complexity_Score'] * 0.3) +
    (df['Combined_Constraint_Score'] * 0.2)
).round(3)

# Flag: high cost AND high risk
df['Cost_Risk_Flag'] = (
    (df['Material_Cost_USD'] > df['Material_Cost_USD'].quantile(0.75)) &
    (df['Risk_Level'] == 'High')
)

# Flag: high complexity but labelled Low risk (potential mislabelling)
df['Underestimated_Risk_Flag'] = (
    (df['Complexity_Score'] > 0.7) &
    (df['Risk_Level'] == 'Low')
)

# Flag: high dependency count AND long duration (schedule critical)
df['Schedule_Critical_Flag'] = (
    (df['Dependency_Count'] >= 3) &
    (df['Task_Duration_Days'] > df['Task_Duration_Days'].quantile(0.75))
)

print('--- TOP 10 RISKIEST TASKS ---')
top_risk = df[['Task_ID','Risk_Level','Complexity_Score',
               'Combined_Constraint_Score','Composite_Risk_Index']]\
    .sort_values('Composite_Risk_Index', ascending=False).head(10)
print(top_risk.to_string(index=False))

print()
print('--- FLAGS SUMMARY ---')
print(f'High cost + High risk tasks:       {df["Cost_Risk_Flag"].sum()}')
print(f'Underestimated risk tasks:         {df["Underestimated_Risk_Flag"].sum()}')
print(f'Schedule critical tasks:           {df["Schedule_Critical_Flag"].sum()}')

print()
print('--- RISK BREAKDOWN BY LEVEL ---')
print(df.groupby('Risk_Level').agg(
    Task_Count=('Task_ID','count'),
    Avg_Duration=('Task_Duration_Days','mean'),
    Avg_Cost=('Material_Cost_USD','mean'),
    Avg_Complexity=('Complexity_Score','mean')
).round(2).to_string())

--- TOP 10 RISKIEST TASKS ---
Task_ID Risk_Level  Complexity_Score  Combined_Constraint_Score  Composite_Risk_Index
   T503       High              0.89                       0.83                  0.93
   T498       High              0.88                       0.81                  0.93
   T284       High              0.89                       0.78                  0.92
  T1046       High              0.81                       0.86                  0.92
   T144       High              0.82                       0.80                  0.91
   T247       High              0.80                       0.82                  0.90
   T280       High              0.72                       0.92                  0.90
   T677       High              0.87                       0.68                  0.90
  T1258       High              0.74                       0.84                  0.89
   T652       High              0.67                       0.91                  0.88

--- FLAGS SUMMARY ---
H

---
## Stage 4 — Critical Task Flagging

Combine all risk signals into a single **Priority Score** and assign each task a tier. 
Tasks flagged across all three risk dimensions simultaneously are marked as **Triple Threat**.

| Tier | Score | Action |
|---|---|---|
| P1 - Immediate Attention | ≥ 0.70 | Review with PM this week |
| P2 - Monitor Closely | ≥ 0.50 | Weekly check-ins |
| P3 - Standard Review | ≥ 0.30 | Normal tracking |
| P4 - Low Priority | < 0.30 | Minimal oversight |


In [34]:
# Priority score — final ranking combining all risk signals
df['Priority_Score'] = (
    (df['Composite_Risk_Index'] * 0.4) +
    (df['Complexity_Score'] * 0.25) +
    (df['Cost_Per_Day'] / df['Cost_Per_Day'].max() * 0.2) +
    (df['Dependency_Count'] / df['Dependency_Count'].max() * 0.15)
).round(3)

# Assign priority tier
def assign_tier(score):
    if score >= 0.7:   return 'P1 - Immediate Attention'
    elif score >= 0.5: return 'P2 - Monitor Closely'
    elif score >= 0.3: return 'P3 - Standard Review'
    else:              return 'P4 - Low Priority'

df['Priority_Tier'] = df['Priority_Score'].apply(assign_tier)

# Triple threat — flagged on cost, schedule, and composite risk simultaneously
df['Triple_Threat'] = (
    df['Cost_Risk_Flag'] &
    df['Schedule_Critical_Flag'] &
    (df['Composite_Risk_Index'] > 0.7)
)

print('--- TOP 15 CRITICAL TASKS ---')
print(df[['Task_ID','Risk_Level','Task_Duration_Days','Dependency_Count',
          'Material_Cost_USD','Priority_Score','Priority_Tier']]
      .sort_values('Priority_Score', ascending=False)
      .head(15).to_string(index=False))

print()
print('--- PRIORITY TIER BREAKDOWN ---')
tier_order = ['P1 - Immediate Attention','P2 - Monitor Closely',
              'P3 - Standard Review','P4 - Low Priority']
for tier in tier_order:
    count = (df['Priority_Tier'] == tier).sum()
    pct = round(count / len(df) * 100, 1)
    print(f'{tier:<30} {count:>4} tasks ({pct}%)')

print()
print('--- TRIPLE THREAT TASKS ---')
triple = df[df['Triple_Threat']][['Task_ID','Risk_Level','Task_Duration_Days',
                                   'Dependency_Count','Material_Cost_USD','Priority_Score']]
print(f'Total: {len(triple)} tasks')
print(triple.sort_values('Priority_Score', ascending=False).to_string(index=False))

--- TOP 15 CRITICAL TASKS ---
Task_ID Risk_Level  Task_Duration_Days  Dependency_Count  Material_Cost_USD  Priority_Score            Priority_Tier
   T503       High                  75                 4           61223.63            0.75 P1 - Immediate Attention
   T284       High                  80                 4           17568.35            0.74 P1 - Immediate Attention
   T498       High                  74                 4           45454.84            0.74 P1 - Immediate Attention
   T677       High                  82                 4           71760.57            0.73 P1 - Immediate Attention
  T1046       High                  56                 4           50314.86            0.72 P1 - Immediate Attention
   T144       High                  63                 4           66506.74            0.72 P1 - Immediate Attention
   T830       High                   1                 3           80737.80            0.71 P1 - Immediate Attention
   T810       High                

---
## Stage 5 — Summary Report & Export

Produce a plain-text summary report readable by a project manager, 
and export the enriched dataset with all 22 columns as a CSV.


In [35]:
lines = []
lines.append('=' * 60)
lines.append('  CONSTRUCTION PROJECT RISK PIPELINE — SUMMARY REPORT')
lines.append('=' * 60)

lines.append('\n📋 DATASET OVERVIEW')
lines.append(f'  Total tasks analysed:        {len(df):,}')
lines.append(f'  Total material cost:         ${df["Material_Cost_USD"].sum():,.0f}')
lines.append(f'  Avg task duration:           {df["Task_Duration_Days"].mean():.1f} days')
lines.append(f'  Avg dependencies per task:   {df["Dependency_Count"].mean():.1f}')

lines.append('\n⚠️  RISK BREAKDOWN')
for level in ['High', 'Medium', 'Low']:
    count = (df['Risk_Level'] == level).sum()
    pct = round(count / len(df) * 100, 1)
    avg_cost = df[df['Risk_Level'] == level]['Material_Cost_USD'].mean()
    lines.append(f'  {level:<8} {count:>4} tasks ({pct}%)   avg cost ${avg_cost:,.0f}')

lines.append('\n🚨 CRITICAL FLAGS')
lines.append(f'  P1 tasks (immediate attention):    {(df["Priority_Tier"] == "P1 - Immediate Attention").sum()}')
lines.append(f'  Triple threat tasks:               {df["Triple_Threat"].sum()}')
lines.append(f'  High cost + High risk:             {df["Cost_Risk_Flag"].sum()}')
lines.append(f'  Underestimated risk tasks:         {df["Underestimated_Risk_Flag"].sum()}')
lines.append(f'  Schedule critical tasks:           {df["Schedule_Critical_Flag"].sum()}')

lines.append('\n🔴 TOP 5 PRIORITY TASKS')
top5 = df.nlargest(5, 'Priority_Score')[['Task_ID','Risk_Level',
        'Task_Duration_Days','Material_Cost_USD','Priority_Score']]
for _, row in top5.iterrows():
    lines.append(f'  {row["Task_ID"]:<8} | {row["Risk_Level"]:<6} | '
                 f'{int(row["Task_Duration_Days"])} days | '
                 f'${row["Material_Cost_USD"]:>10,.0f} | '
                 f'Priority: {row["Priority_Score"]}')

lines.append('\n🟡 UNDERESTIMATED RISK SAMPLE (labelled Low, high complexity)')
under = df[df['Underestimated_Risk_Flag']].nlargest(5, 'Complexity_Score')[[
    'Task_ID','Complexity_Score','Combined_Constraint_Score','Dependency_Count']]
for _, row in under.iterrows():
    lines.append(f'  {row["Task_ID"]:<8} | complexity {row["Complexity_Score"]} | '
                 f'constraint {row["Combined_Constraint_Score"]} | '
                 f'{int(row["Dependency_Count"])} dependencies')

lines.append('\n' + '=' * 60)
lines.append('  Pipeline: Validation → Feature Engineering → Risk Analysis')
lines.append('  → Critical Flagging → Reporting')
lines.append('  Original columns: 10   Final columns: ' + str(len(df.columns)))
lines.append('=' * 60)

report = '\n'.join(lines)
print(report)

# Save report
with open(OUTPUT_REPORT, 'w') as f:
    f.write(report)

# Export enriched dataset
df.to_csv(OUTPUT_CSV, index=False)

print(f'\n✅ Report saved:  {OUTPUT_REPORT}')
print(f'✅ Dataset saved: {OUTPUT_CSV}')
print(f'   Columns: 10 → {len(df.columns)}')

  CONSTRUCTION PROJECT RISK PIPELINE — SUMMARY REPORT

📋 DATASET OVERVIEW
  Total tasks analysed:        1,300
  Total material cost:         $64,382,642
  Avg task duration:           43.4 days
  Avg dependencies per task:   2.0

⚠️  RISK BREAKDOWN
  High      263 tasks (20.2%)   avg cost $48,411
  Medium    377 tasks (29.0%)   avg cost $49,905
  Low       660 tasks (50.8%)   avg cost $49,752

🚨 CRITICAL FLAGS
  P1 tasks (immediate attention):    8
  Triple threat tasks:               5
  High cost + High risk:             64
  Underestimated risk tasks:         87
  Schedule critical tasks:           128

🔴 TOP 5 PRIORITY TASKS
  T503     | High   | 75 days | $    61,224 | Priority: 0.747
  T284     | High   | 80 days | $    17,568 | Priority: 0.743
  T498     | High   | 74 days | $    45,455 | Priority: 0.74
  T677     | High   | 82 days | $    71,761 | Priority: 0.728
  T1046    | High   | 56 days | $    50,315 | Priority: 0.721

🟡 UNDERESTIMATED RISK SAMPLE (labelled Low, high com

---
## Summary

This pipeline transformed a raw 10-column construction dataset into a 
22-column enriched output with actionable risk intelligence.

### Key findings
- **$64.4M** total material cost across 1,300 tasks
- **8 P1 tasks** requiring immediate project manager attention
- **87 tasks** carry higher complexity than their Low risk label suggests
- **5 triple threat tasks** are simultaneously expensive, high risk, and schedule critical
- Top priority task **T503** scores 0.747 — 75 days, 4 dependencies, $61K cost

### Skills demonstrated
- pandas data pipeline architecture
- Schema validation with assertions
- Feature engineering and composite scoring
- Risk classification and flag logic
- Structured reporting for non-technical stakeholders
